### Workflow
#### Preparation

install the RNA-FM package
load the necessary libraries
#### Task 1. RNA family clustering

Goal: to demonstrate that RNA-FM embeddings are biologically meaningful

read RNA sequences for each family from FASTA files
generate the RNA-FM embeddings for each sequence
t-SNE dimension reduction on the generated embeddings
plot the embeddings in the 2D space

#### Task 2. RNA type classification

Goal: to demonstrate how to use RNA-FM for downstream applications

read RNA sequences for each type from a FASTA file
generate RNA-FM embeddings for each sequence
build the dataset and model
train and validate the model
test the model on a dataset excluded from training

In [2]:
# Toy run from RNA-FM's git repo -- just to see what actually goes into the model.
import argparse
from pathlib import Path

import torch
import fm

# rna-fm 0.2.2 calls torch.load() without weights_only=False. PyTorch flipped that
# default to True in 2.6, and this checkpoint carries an argparse.Namespace, so on
# torch 2.13 the load raises UnpicklingError. Allowlist just that one class.
# Do NOT "fix" it with weights_only=False -- that re-enables arbitrary code execution.
torch.serialization.add_safe_globals([argparse.Namespace])

# 1. Load RNA-FM model from the local checkpoint.
# The path has to go to rna_fm_t12(), not to get_batch_converter(). Without a valid
# path fm silently falls back to re-downloading 1.19 GB from CUHK, so assert first.
# This notebook lives in probing/, so anchor paths to the repo root rather than cwd.
def repo_root(start=None):
    p = (start or Path.cwd()).resolve()
    for cand in (p, *p.parents):
        if (cand / "datasets_v2.csv").exists():
            return cand
    raise FileNotFoundError("repo root not found (looked for datasets_v2.csv)")

REPO = repo_root()
WEIGHTS = REPO / "RNA-FM_stuff" / "pre-trained-models" / "RNA-FM_pretrained.pth"
assert WEIGHTS.exists(), f"checkpoint missing: {WEIGHTS} -- see RNA-FM_stuff/SOURCE.md"

model, alphabet = fm.pretrained.rna_fm_t12(str(WEIGHTS))
batch_converter = alphabet.get_batch_converter()
model.eval()  # disables dropout for deterministic results

# 2. Prepare data
data = [
    ("RNA1", "GGGUGCGAUCAUACCAGCACUAAUGCCCUCCUGGGAAGUCCUCGUGUUGCACCCCU"),
    ("RNA2", "GGGUGUCGCUCAGUUGGUAGAGUGCUUGCCUGGCAUGCAAGAAACCUUGGUUCAAUCCCCAGCACUGCA"),
    ("RNA3", "CGAUUCNCGUUCCC--CCGCCUCCA"),
]
batch_labels, batch_strs, batch_tokens = batch_converter(data)

# 3. Extract embeddings (on CPU)
with torch.no_grad():
    results = model(batch_tokens, repr_layers=[12])
token_embeddings = results["representations"][12]
print(token_embeddings.shape)
# -> torch.Size([3, 71, 640]) = (batch, longest_len + 2, 640).
# Why 71: the batch is padded to the longest sequence, RNA2 at 69 nt, plus one BOS
# and one EOS token. So index 0 and index -1 are not nucleotides -- strip them before
# pooling, and mask the padding on the shorter sequences (RNA1 56 nt, RNA3 25 nt).


torch.Size([3, 71, 640])


In [4]:
print(batch_labels)
print(batch_strs)
print(batch_tokens)

['RNA1', 'RNA2', 'RNA3']
['GGGUGCGAUCAUACCAGCACUAAUGCCCUCCUGGGAAGUCCUCGUGUUGCACCCCU', 'GGGUGUCGCUCAGUUGGUAGAGUGCUUGCCUGGCAUGCAAGAAACCUUGGUUCAAUCCCCAGCACUGCA', 'CGAUUCNCGUUCCC--CCGCCUCCA']
tensor([[ 0,  6,  6,  6,  7,  6,  5,  6,  4,  7,  5,  4,  7,  4,  5,  5,  4,  6,
          5,  4,  5,  7,  4,  4,  7,  6,  5,  5,  5,  7,  5,  5,  7,  6,  6,  6,
          4,  4,  6,  7,  5,  5,  7,  5,  6,  7,  6,  7,  7,  6,  5,  4,  5,  5,
          5,  5,  7,  2,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
        [ 0,  6,  6,  6,  7,  6,  7,  5,  6,  5,  7,  5,  4,  6,  7,  7,  6,  6,
          7,  4,  6,  4,  6,  7,  6,  5,  7,  7,  6,  5,  5,  7,  6,  6,  5,  4,
          7,  6,  5,  4,  4,  6,  4,  4,  4,  5,  5,  7,  7,  6,  6,  7,  7,  5,
          4,  4,  7,  5,  5,  5,  5,  4,  6,  5,  4,  5,  7,  6,  5,  4,  2],
        [ 0,  5,  6,  4,  7,  7,  5, 18,  5,  6,  7,  7,  5,  5,  5, 19, 19,  5,
          5,  6,  5,  5,  7,  5,  5,  4,  2,  1,  1,  1,  1,  1,  1,  1,  1,  1,
        